# Asistente — Detección temprana de salud mental

Proyecto DL/NLP: BiLSTM (from scratch) + BETO (fine-tuning) + Whisper (voz).

**Archivos del proyecto:**
- `asistente.py` — lógica del asistente (inferencia, diálogo, BiLSTM)
- `demo_ui.py` — interfaz Gradio (entrevista 4 turnos + voz)
- `requirements.txt` — dependencias (`pip install -r requirements.txt`)
- `outputs/train.csv`, `val.csv`, `test.csv` — datos de entrenamiento

Ejecutar celdas **en orden**. Para la exposición: entrenar (celdas 1–6) y lanzar la **última celda** (demo Gradio).

## 1. Configuración

In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, DataCollatorWithPadding

from asistente import (
    BiLSTMClasificador, TextoDataset, balancear_train_df, construir_vocab,
    CARPETA_SALIDA, CARPETA_MODELO, CARPETA_BILSTM, NOMBRE_MODELO, LONGITUD_MAX,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dispositivo)

# True = entrenar (~25-40 min BETO en GPU). False = cargar modelo ya guardado.
ENTRENAR_BILSTM = True
ENTRENAR_BETO = True
BALANCEAR_TRAIN = True  # oversampling clases minoritarias (otro, trauma, sustancias)

EPOCAS_BETO = 4
TAMANO_LOTE = 16
TASA_APRENDIZAJE = 2e-5
LABEL_SMOOTHING = 0.1
MAX_VOCAB = 20000
MAX_LEN_BILSTM = 256
EMBED_DIM = 128
HIDDEN_DIM = 256
EPOCAS_BILSTM = 10
BATCH_BILSTM = 64
PATIENCIA_BILSTM = 3
LR_BILSTM = 5e-4

## 2. Datos

In [ ]:
train_df = pd.read_csv(CARPETA_SALIDA / "train.csv")
val_df = pd.read_csv(CARPETA_SALIDA / "val.csv")
test_df = pd.read_csv(CARPETA_SALIDA / "test.csv")

print(f"Train original {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")
print("Distribución original:\n", train_df["label"].value_counts())

etiquetas = sorted(train_df["label"].unique())
id_a_etiqueta = {i: e for i, e in enumerate(etiquetas)}
etiqueta_a_id = {e: i for i, e in enumerate(etiquetas)}
num_clases = len(etiquetas)

if BALANCEAR_TRAIN:
    train_df_modelo = balancear_train_df(train_df, columna="label", semilla=SEED)
    print(f"\nTrain balanceado: {len(train_df_modelo)} muestras")
    print(train_df_modelo["label"].value_counts())
else:
    train_df_modelo = train_df.copy()

y_train_ids = train_df_modelo["label"].map(etiqueta_a_id).values
pesos_clase = compute_class_weight("balanced", classes=np.arange(num_clases), y=y_train_ids)
pesos_clase_tensor = torch.tensor(pesos_clase, dtype=torch.float32).to(dispositivo)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
train_df["label"].value_counts().plot(kind="bar", ax=axes[0], title="Train original")
train_df_modelo["label"].value_counts().plot(kind="bar", ax=axes[1], title="Train para entrenamiento")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 3. Baseline (TF-IDF + regresión logística)

In [ ]:
vectorizador = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vectorizador.fit_transform(train_df_modelo["texto"])
X_val = vectorizador.transform(val_df["texto"])
X_test = vectorizador.transform(test_df["texto"])

y_train = train_df_modelo["label"].map(etiqueta_a_id)
y_val = val_df["label"].map(etiqueta_a_id)
y_test = test_df["label"].map(etiqueta_a_id)

baseline = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
baseline.fit(X_train, y_train)
f1_baseline_test = f1_score(y_test, baseline.predict(X_test), average="macro")
print(f"F1 macro baseline (test): {f1_baseline_test:.4f}")

## 4. BiLSTM from scratch

In [ ]:
vocab = construir_vocab(train_df_modelo["texto"], MAX_VOCAB)
ds_train = TextoDataset(train_df_modelo["texto"], y_train.tolist(), vocab, MAX_LEN_BILSTM)
ds_val = TextoDataset(val_df["texto"], y_val.tolist(), vocab, MAX_LEN_BILSTM)
loader_train = DataLoader(ds_train, batch_size=BATCH_BILSTM, shuffle=True)
loader_val = DataLoader(ds_val, batch_size=BATCH_BILSTM)

modelo_bilstm = BiLSTMClasificador(len(vocab), EMBED_DIM, HIDDEN_DIM, num_clases).to(dispositivo)
criterio = nn.CrossEntropyLoss(weight=pesos_clase_tensor)
optimizador = torch.optim.AdamW(modelo_bilstm.parameters(), lr=LR_BILSTM, weight_decay=1e-4)

mejor_f1_bilstm = 0.0
epocas_sin_mejora = 0
CARPETA_BILSTM.mkdir(parents=True, exist_ok=True)
ruta_mejor_bilstm = CARPETA_BILSTM / "bilstm.pt"

if ENTRENAR_BILSTM:
    print(f"Entrenando BiLSTM ({EPOCAS_BILSTM} épocas max, early stopping={PATIENCIA_BILSTM})...")
    for epoca in range(1, EPOCAS_BILSTM + 1):
        modelo_bilstm.train()
        for x, y in loader_train:
            x, y = x.to(dispositivo), torch.tensor(y, dtype=torch.long).to(dispositivo)
            optimizador.zero_grad()
            loss = criterio(modelo_bilstm(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(modelo_bilstm.parameters(), 1.0)
            optimizador.step()
        modelo_bilstm.eval()
        preds, reals = [], []
        with torch.no_grad():
            for x, y in loader_val:
                preds.extend(modelo_bilstm(x.to(dispositivo)).argmax(1).cpu().tolist())
                reals.extend(y if isinstance(y, list) else y.tolist())
        f1_val = f1_score(reals, preds, average="macro")
        print(f"Época {epoca} — F1 val: {f1_val:.4f}")
        if f1_val > mejor_f1_bilstm:
            mejor_f1_bilstm = f1_val
            epocas_sin_mejora = 0
            torch.save(modelo_bilstm.state_dict(), ruta_mejor_bilstm)
            print("  → Mejor modelo guardado")
        else:
            epocas_sin_mejora += 1
            if epocas_sin_mejora >= PATIENCIA_BILSTM:
                print(f"Early stopping en época {epoca}")
                break
else:
    print("ENTRENAR_BILSTM=False — cargando pesos guardados")
    ckpt = ruta_mejor_bilstm
    try:
        modelo_bilstm.load_state_dict(torch.load(ckpt, map_location=dispositivo, weights_only=True))
    except TypeError:
        modelo_bilstm.load_state_dict(torch.load(ckpt, map_location=dispositivo))

if ruta_mejor_bilstm.exists():
    try:
        modelo_bilstm.load_state_dict(torch.load(ruta_mejor_bilstm, map_location=dispositivo, weights_only=True))
    except TypeError:
        modelo_bilstm.load_state_dict(torch.load(ruta_mejor_bilstm, map_location=dispositivo))

modelo_bilstm.eval()
loader_test = DataLoader(TextoDataset(test_df["texto"], y_test.tolist(), vocab, MAX_LEN_BILSTM), batch_size=128)
preds_bilstm = []
with torch.no_grad():
    for x, _ in loader_test:
        preds_bilstm.extend(modelo_bilstm(x.to(dispositivo)).argmax(1).cpu().tolist())
f1_bilstm_test = f1_score(y_test, preds_bilstm, average="macro")
print(f"F1 macro BiLSTM (test): {f1_bilstm_test:.4f}")

## 5. BETO fine-tuning

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(NOMBRE_MODELO)

def a_dataset(df):
    return Dataset.from_dict({"text": df["texto"].tolist(), "labels": df["label"].map(etiqueta_a_id).tolist()})

ds_train = a_dataset(train_df_modelo).map(
    lambda b: tokenizer(b["text"], truncation=True, max_length=LONGITUD_MAX), batched=True,
)
ds_val = a_dataset(val_df).map(
    lambda b: tokenizer(b["text"], truncation=True, max_length=LONGITUD_MAX), batched=True,
)

modelo = AutoModelForSequenceClassification.from_pretrained(
    NOMBRE_MODELO, num_labels=num_clases, id2label=id_a_etiqueta, label2id=etiqueta_a_id,
)

from transformers import Trainer, EarlyStoppingCallback

class TrainerPonderado(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(
            weight=pesos_clase_tensor, label_smoothing=LABEL_SMOOTHING,
        )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir=str(CARPETA_MODELO),
    num_train_epochs=EPOCAS_BETO,
    per_device_train_batch_size=TAMANO_LOTE,
    per_device_eval_batch_size=TAMANO_LOTE * 2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    learning_rate=TASA_APRENDIZAJE,
    weight_decay=0.05,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=(dispositivo == "cuda"),
    save_total_limit=2,
    report_to="none",
)

def metricas(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"f1_macro": f1_score(p.label_ids, preds, average="macro")}

entrenador = TrainerPonderado(
    model=modelo, args=args, train_dataset=ds_train, eval_dataset=ds_val,
    processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=metricas,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

if ENTRENAR_BETO:
    print(f"Entrenando BETO ({EPOCAS_BETO} épocas, batch {TAMANO_LOTE}, train balanceado)...")
    entrenador.train()
    entrenador.save_model(str(CARPETA_MODELO))
    tokenizer.save_pretrained(str(CARPETA_MODELO))
    print("Modelo guardado en", CARPETA_MODELO)
else:
    print("ENTRENAR_BETO=False — usando modelo guardado")
    print("Ruta:", CARPETA_MODELO)

## 6. Comparación de modelos

In [ ]:
from asistente import cargar_beto
import seaborn as sns

modelo_eval, tok_eval = cargar_beto()
textos_test = test_df["texto"].tolist()
y_true = y_test.values

preds_beto = []
for i in range(0, len(textos_test), 32):
    lote = textos_test[i:i+32]
    ent = tok_eval(lote, return_tensors="pt", truncation=True, max_length=LONGITUD_MAX, padding=True)
    ent = {k: v.to(dispositivo) for k, v in ent.items()}
    with torch.no_grad():
        preds_beto.extend(modelo_eval(**ent).logits.argmax(1).cpu().tolist())

f1_beto_test = f1_score(y_true, preds_beto, average="macro")

resultados = pd.DataFrame({
    "Modelo": ["Baseline", "BiLSTM", "BETO"],
    "F1 macro": [f1_baseline_test, f1_bilstm_test, f1_beto_test],
}).sort_values("F1 macro", ascending=False)
print(resultados.to_string(index=False))

resultados.plot(x="Modelo", y="F1 macro", kind="bar", legend=False, figsize=(6, 3), color="teal")
plt.ylabel("F1 macro")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

print("\nReporte BETO (test):")
print(classification_report(y_true, preds_beto, target_names=etiquetas, zero_division=0))

cm = confusion_matrix(y_true, preds_beto)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=etiquetas, yticklabels=etiquetas, cmap="Blues")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de confusión — BETO (test)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Prueba rápida en notebook

In [ ]:
from asistente import procesar_mensaje, resumen_analisis

texto = "Tengo demasiadas tareas y no puedo dormir. Me siento muy ansioso."
r = procesar_mensaje(texto)
print("Clase:", r["clase"], f"({r['confianza']:.1f}%)")
print("Respuesta:", r["respuesta"])
print("\n--- Panel demo ---")
print(resumen_analisis(r))

## 8. Demo en vivo

Ejecuta la celda → abre en **Chrome/Edge**. Pulsa **«Grabar (6 s)»** o escribe y **Enviar**. La voz suena en tus altavoces.

In [ ]:
import importlib
import asistente
import demo_ui

importlib.reload(asistente)
importlib.reload(demo_ui)

demo_ui.lanzar_demo(inline=False, inbrowser=True)